[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-05-human-in-the-loop.ipynb#scrollTo=11223344)

---
# Day 5 · Human-in-the-Loop Breakpoints and Resumption
**certified-journeys / ai-agents-certified** · Day 5 · HITL

> **Goal for today:** Build a graph that pauses before a dangerous action, shows the pending tool call to the user, and resumes from an exact state snapshot using the same `thread_id` — with both interrupt-based and Command-based resumption patterns.


In [ ]:
%pip install -q langgraph langchain-openai


## Step 1 · Why Human-in-the-Loop?

Autonomous agents are powerful — but some actions are irreversible: deleting records, sending emails, transferring funds. HITL adds a mandatory review gate before those actions execute.

| Mechanism | How it works | Best for |
|-----------|-------------|----------|
| `interrupt_before` | Graph pauses **before** a named node runs | Review a pending tool call |
| `interrupt_after` | Graph pauses **after** a named node runs | Review the tool's output before the LLM sees it |
| `Command(resume=…)` | Programmatic resumption with a value injected into state | Automated approval workflows |

**Key insight:** The graph state is fully serialised into a **checkpoint** at the interrupt point. Resumption reads that checkpoint — the graph never re-runs the steps before the interrupt.


In [ ]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver  # in-memory checkpointer

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# Simulate a dangerous tool — deleting a database record
@tool
def delete_record(record_id: str) -> str:
    """Delete a record from the database by its ID. This action is irreversible."""
    # In a real system this would execute DELETE FROM table WHERE id = record_id
    return f"Record {record_id} has been permanently deleted."


@tool
def lookup_record(record_id: str) -> str:
    """Look up a record in the database by its ID. Read-only."""
    return f"Record {record_id}: {{name: 'Alice', email: 'alice@example.com', balance: 1500.00}}"


tools = [lookup_record, delete_record]
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)

print("Tools defined:", [t.name for t in tools])


**What just happened?**
- `MemorySaver` is an in-process checkpointer that stores state in a Python dict — perfect for development and testing.
- For production, swap `MemorySaver` for `SqliteSaver` (Day 6) or `PostgresSaver`.
- We define two tools: a safe read (`lookup_record`) and a dangerous write (`delete_record`).
- **The interrupt will fire before the `tools` node**, so the user sees the pending `delete_record` call before it executes.


## Step 2 · Build the Graph with `interrupt_before`

`interrupt_before=["tools"]` tells LangGraph to pause **every time** it is about to enter the `tools` node. The graph saves a checkpoint and raises a `GraphInterrupt`. The caller catches this and can inspect state before deciding to resume or abort.


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"


builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")

# MemorySaver is required for interrupt to work — graph must have a checkpointer
memory = MemorySaver()

# interrupt_before=["tools"] — pause before every tools node execution
graph = builder.compile(checkpointer=memory, interrupt_before=["tools"])

print("Graph compiled with interrupt_before=['tools']")


**What just happened?**
- `checkpointer=memory` is **mandatory** for interrupts — LangGraph needs somewhere to serialise the paused state.
- `interrupt_before=["tools"]` is a list of node names. The graph will pause before ANY of those nodes.
- The graph is otherwise identical to Day 4 — the interrupt is purely a compile-time flag.
- **Without a checkpointer**, `interrupt_before` raises a `ValueError` at compile time.


## Step 3 · Trigger the Interrupt and Inspect Pending State

We run the graph until it pauses, then print the pending tool call so the human can review it before approving.


In [ ]:
# thread_id identifies this conversation — MUST be reused on resumption
thread_id = "hitl-demo-thread-001"
config = {"configurable": {"thread_id": thread_id}}

user_request = HumanMessage(
    content="First look up record user-42, then delete it."
)

print("Starting graph run...")
print("=" * 60)

# The first run will pause before the tools node
# graph.invoke returns state at the interrupt point (not END)
state_at_interrupt = graph.invoke(
    {"messages": [user_request]},
    config=config,
)

# Inspect what the LLM planned to do
last_msg = state_at_interrupt["messages"][-1]
print("Graph paused. Pending action(s):")
print()

if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
    for tc in last_msg.tool_calls:
        danger = "delete" in tc["name"]  # flag dangerous tools
        marker = "⚠️  DANGEROUS" if danger else "✅ SAFE"
        print(f"{marker} | Tool: {tc['name']} | Args: {tc['args']}")
else:
    print("No tool calls found — graph may have reached END already.")

print()
print(f"Thread ID to resume: {thread_id}")


**What just happened?**
- `graph.invoke` returned early — at the interrupt point, not at `END`.
- The returned state is the **snapshot just before** the `tools` node would have run.
- We inspect `last_msg.tool_calls` to show the human exactly what the agent plans to do.
- **`thread_id` is everything.** The checkpoint is keyed by thread_id — lose it and you can't resume.


## Step 4 · Resume from the Checkpoint After Human Approval

Resumption is simple: call `invoke` again with **the same `thread_id`** and **`None` as the input** (indicating "continue from where you left off, no new input").


In [ ]:
# Simulate human approval
human_approved = True  # change to False to see what an abort looks like

if human_approved:
    print("Human approved. Resuming graph...")
    print("=" * 60)

    # None input = resume from checkpoint, don't inject new messages
    resumed_state = graph.invoke(None, config=config)

    # The graph may pause again (if there are more tool calls) or reach END
    final_msg = resumed_state["messages"][-1]
    print("Final agent response:")
    print(final_msg.content)

    # Verify state survived across the interrupt
    print("\nFull message history (proving state preservation):")
    for i, msg in enumerate(resumed_state["messages"]):
        role = type(msg).__name__
        preview = (msg.content or str(getattr(msg, 'tool_calls', '')))[:80]
        print(f"  [{i}] {role}: {preview}")
else:
    print("Human rejected. Aborting — not resuming the graph.")
    print("The checkpoint still exists and can be resumed later.")


**What just happened?**
- `graph.invoke(None, config=config)` is the canonical resumption call — `None` means "use the saved checkpoint".
- **The full message history is preserved.** The resumed graph sees all messages from before the interrupt.
- If the graph hits another tool call after resumption, it will pause again (because `interrupt_before` fires every time).
- **Aborting is as simple as not calling resume** — the checkpoint stays in memory but the agent never continues.


## Step 5 · Verify State Preservation with `get_state`

`graph.get_state(config)` lets you inspect the saved snapshot without resuming. This is how you build monitoring dashboards and audit trails.


In [ ]:
# After a completed run we can inspect the final checkpoint
final_snapshot = graph.get_state(config)

print("Checkpoint metadata:")
print(f"  Thread ID:     {final_snapshot.config['configurable']['thread_id']}")
print(f"  Checkpoint ID: {final_snapshot.config['configurable'].get('checkpoint_id', 'N/A')}")
print(f"  Next node(s):  {final_snapshot.next}")
print(f"  Message count: {len(final_snapshot.values['messages'])}")
print()

# Also list all checkpoints for this thread (one per node execution)
print("All checkpoints for this thread:")
history = list(graph.get_state_history(config))
for i, snap in enumerate(history):
    cid = snap.config['configurable'].get('checkpoint_id', '?')
    msgs = len(snap.values['messages'])
    nxt = snap.next
    print(f"  [{i}] checkpoint_id={cid[:16]}… | messages={msgs} | next={nxt}")


**What just happened?**
- `get_state` returns a `StateSnapshot` with `.values` (the state dict), `.next` (which node runs next), and `.config` (the thread + checkpoint IDs).
- `get_state_history` returns **one snapshot per node execution** — this is LangGraph's time-travel capability.
- **`next = ()`** means the graph has reached `END`.
- You can resume from any historical snapshot by passing its `checkpoint_id` in config: `{"configurable": {"thread_id": ..., "checkpoint_id": ...}}`.


## Step 6 · Command-Based Resumption

`Command(resume=value)` is an alternative to `None`-input resumption. It lets you inject a value into the interrupt point — useful for approval workflows where the human's decision should influence subsequent steps.


In [ ]:
from langgraph.types import Command

# Build a new graph where the agent node uses interrupt() explicitly
# This gives finer control — the interrupt fires inside a node, not at graph edges
from langgraph.types import interrupt as lg_interrupt


class ApprovalState(TypedDict):
    messages: Annotated[list, add_messages]
    approval_decision: str  # "approved" or "rejected" — injected via Command


def approval_agent_node(state: ApprovalState) -> dict:
    """Agent node that interrupts itself to request human approval."""
    response = llm_with_tools.invoke(state["messages"])

    # If the LLM wants to call delete_record, pause and ask the human
    if response.tool_calls and any(tc["name"] == "delete_record" for tc in response.tool_calls):
        tc_summary = [{"tool": tc["name"], "args": tc["args"]} for tc in response.tool_calls]
        # lg_interrupt pauses execution here and returns the value to the caller
        # When resumed with Command(resume=decision), 'decision' is the return value
        decision = lg_interrupt({"pending_action": tc_summary, "question": "Approve this deletion?"})
        # Execution resumes HERE with decision = whatever Command(resume=...) passed
        if decision != "approved":
            # Inject a refusal message instead of executing the tool
            from langchain_core.messages import AIMessage as AI
            return {
                "messages": [AI(content="Action rejected by human reviewer. Deletion aborted.")],
                "approval_decision": "rejected",
            }

    return {"messages": [response], "approval_decision": state.get("approval_decision", "")}


def approval_should_continue(state: ApprovalState) -> str:
    last = state["messages"][-1]
    # Don't call tools if the decision was rejection
    if state.get("approval_decision") == "rejected":
        return "end"
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"


builder2 = StateGraph(ApprovalState)
builder2.add_node("agent", approval_agent_node)
builder2.add_node("tools", tool_node)
builder2.add_edge(START, "agent")
builder2.add_conditional_edges("agent", approval_should_continue, {"tools": "tools", "end": END})
builder2.add_edge("tools", "agent")

memory2 = MemorySaver()
graph2 = builder2.compile(checkpointer=memory2)

print("Command-based HITL graph compiled")


In [ ]:
thread2 = "hitl-command-thread-001"
config2 = {"configurable": {"thread_id": thread2}}

print("--- Run 1: trigger the interrupt ---")
partial = graph2.invoke(
    {"messages": [HumanMessage(content="Delete record user-99")], "approval_decision": ""},
    config=config2,
)

# partial.get('__interrupt__') holds the interrupt payload when using lg_interrupt()
snapshot = graph2.get_state(config2)
print("Graph paused. Next node:", snapshot.next)
print("Interrupt payload:", snapshot.tasks[0].interrupts if snapshot.tasks else "N/A")

print()
print("--- Run 2: resume with Command(resume='approved') ---")
# Command(resume=value) injects 'value' as the return from lg_interrupt()
approved_result = graph2.invoke(Command(resume="approved"), config=config2)
print("Final message:", approved_result["messages"][-1].content)

# --- Demonstrate rejection on a new thread ---
print()
print("--- New thread: resume with rejection ---")
thread3 = "hitl-command-thread-002"
config3 = {"configurable": {"thread_id": thread3}}
graph2.invoke(
    {"messages": [HumanMessage(content="Delete record user-77")], "approval_decision": ""},
    config=config3,
)
rejected_result = graph2.invoke(Command(resume="rejected"), config=config3)
print("Final message:", rejected_result["messages"][-1].content)


**What just happened?**
- `lg_interrupt(payload)` pauses execution mid-node and exposes `payload` to the caller via `snapshot.tasks[0].interrupts`.
- `Command(resume="approved")` injects `"approved"` as the return value of `lg_interrupt()` — the node resumes from that exact line.
- **Rejection path:** the node received `"rejected"` from `Command`, bypassed the tool call, and returned a refusal `AIMessage`.
- This pattern is more flexible than `interrupt_before` — you can interrupt mid-node based on runtime conditions.


## Step 7 · `interrupt_after` — Review Tool Output Before the LLM Sees It

`interrupt_after=["tools"]` pauses **after** the tool runs but **before** the LLM processes the result. This lets you redact or modify tool outputs.


In [ ]:
memory3 = MemorySaver()
graph3 = builder.compile(checkpointer=memory3, interrupt_after=["tools"])

thread4 = "interrupt-after-demo"
config4 = {"configurable": {"thread_id": thread4}}

print("Run 1: agent calls lookup_record, pauses AFTER tool runs")
state1 = graph3.invoke(
    {"messages": [HumanMessage(content="Look up record user-42, then tell me the email address.")]},
    config=config4,
)

snap = graph3.get_state(config4)
print("Next node after interrupt:", snap.next)

# Inspect the tool output that will be fed to the LLM
last_tool_msg = next(
    (m for m in reversed(state1["messages"]) if isinstance(m, ToolMessage)), None
)
if last_tool_msg:
    print("Tool output (the LLM will receive this):", last_tool_msg.content)

print()
print("Resuming — LLM will now process the tool output")
state2 = graph3.invoke(None, config=config4)
print("Final answer:", state2["messages"][-1].content)


**What just happened?**
- `interrupt_after=["tools"]` fires after the `ToolMessage` is added to state, before the next `agent` node run.
- You can **modify the tool message in state** using `graph.update_state(config, ...)` before resuming — useful for redacting PII or fixing malformed tool output.
- This is the **data review** pattern: the human is a quality gate between the tool and the LLM's interpretation.


In [ ]:
# Challenge: Build a HITL graph for a payment agent
#
# Requirements:
#   1. Define a tool: transfer_funds(from_account: str, to_account: str, amount: float) -> str
#      that returns a confirmation string (simulate the transfer, no real API needed)
#   2. Build a graph with interrupt_before=["tools"] and MemorySaver
#   3. Send a request: "Transfer $500 from account A001 to account B002"
#   4. At the interrupt, print the pending transfer details clearly
#   5. Resume with approval and verify the tool ran
#   6. On a NEW thread, run the same request but do NOT resume — verify the checkpoint exists
#      by calling get_state() and confirming next != ()

# Your solution here

# TODO: define transfer_funds tool
# TODO: build StateGraph with interrupt_before
# TODO: run, interrupt, print details
# TODO: resume with approval
# TODO: new thread, no resume, verify checkpoint


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `interrupt_before` | Pause before a node; must have a checkpointer |
| `interrupt_after` | Pause after a node to review tool output |
| `thread_id` | The key to resumption — same ID = same conversation, different ID = fresh run |
| Resume with `None` | `graph.invoke(None, config)` — continue from checkpoint, no new input |
| `Command(resume=val)` | Inject a value at the exact point where `interrupt()` was called |
| `get_state` | Inspect current snapshot without resuming |
| `get_state_history` | All snapshots for a thread — enables time-travel |

> **Tip:** Thread IDs are the key to HITL. If you call invoke with a different thread_id after an interrupt, you get a fresh run, not a resumed one. Always store and reuse the exact same thread_id.

---
## What's next
**Day 6** → Persistence and Checkpointers: swap `MemorySaver` for `SqliteSaver`, inspect raw checkpoint rows in SQLite, and simulate a mid-run crash to prove state survives process restart.

Mark Day 5 complete in your [tracker](../index.html).
